##### Library imports

In [ ]:
import re
import os
import pandas as pd
import cv2 as cv
import SimpleITK as sitk
import numpy as np 
import json
from skimage import measure, filters
import matplotlib.pyplot as plt

##### Function definitions

In [ ]:
def window_stack_sitk(input_im, window_center=40, window_width=80):
    """
    Clamps values outside [min, max] to the edge values.
    
    Parameters
    ----------
    input_im : sitk.Image
        Input, unwindowed, brain image
    window_center : int, optional
        The center of the Hounsfield Unit (HU) scale in the windowed image. Default is 40 HU (brain).
    window_width : int, optional
        The total HU window width around the window_center, in the windowed image. Default is 80 HU (brain).

    Returns
    -------
    windowed_im : sitk.Image
        Windowed CT image with the desired tissue's visualization optimized (e.g., 0 - 80 HU for brain tissue)
    
    """
    img_min = window_center - (window_width // 2)
    img_max = window_center + (window_width // 2)


    windowed_im = sitk.IntensityWindowing(
        input_im, 
        windowMinimum=float(img_min), 
        windowMaximum=float(img_max), 
        outputMinimum=float(img_min), 
        outputMaximum=float(img_max)
    )
    
    return windowed_im

In [ ]:
def getLargestCC(blobs_labels):
    """Returns the largest connected component from an image containing blobs of discrete labels
    
    Parameters
    ----------
    blobs_labels : numpy.ndarray 
        A collection of discrete blobs (e.g., cerebrospinal fluid [CSF] spaces including the lateral ventricles, longitudinal fissure, sulci etc.)

    Returns
    -------
    largestCC: numpy.ndarray
        The largest connected component from a collection of discrete blobs (e.g. lateral ventricles from a collection of CSF spaces)  
    
    """
    # assume at least 1 CC apart from background
    if blobs_labels.max() == 0:
        raise ValueError('Blank segmentation, inspect processing up to here.')
    #this assumes that the background is the largest CC (label 0)
    largestCC = blobs_labels == np.argmax(np.bincount(blobs_labels.flat)[1:])+1 
    return largestCC

In [ ]:
def rotate_image(image, dimension = 3):
    """Rotates NIfTI coronal images so they are upright for visualization. Simple insights tool kit (SITK) utilizes the LPS coordinate system, 
       meaning image voxel coordinates are assumed to increase in the Right --> Left, Anterior --> Posterior, and Inferior --> Superior directions. 
       However, NIfTI images use the RAS coordinate system, which makes coronal slices (vertical cross-sections) appear inverted when read with SITK. 
       This function resamples the image such that it is visualized in upright direction, enabling interpretable visualization and processing. 
    
    Parameters
    ----------
    image : sitk.Image
        The original NIfTI image (Note that the direction cosine should be [-1,0,0,0,-1,0,0,0,1], although this is a coronal image, it is 
        assumed to be resampled and cropped such that the direction cosine corresponds to an axial volume)
    dimension : int, optional
        Dimensionality of the original and rotation-corrected image. Default is 3.
    
    Returns
    -------
    rotated_image : sitk.Image
        Rotation-corrected image, with a direction cosine of [1,0,0,0,1,0,0,0,1]
    """

    transform = sitk.AffineTransform(dimension)
    transform.SetCenter(image.TransformContinuousIndexToPhysicalPoint(np.array(image.GetSize())//2.0))
    s = image.GetSpacing()[2]

    matrix = np.array([[1.0,0.0,0.0],[0.0,1.0,0.0],[0.0,0.0,-1.0]])

   
    transform.SetMatrix(matrix.ravel())

    extreme_points = [image.TransformIndexToPhysicalPoint((0,0,0)), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,0)),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,0,image.GetDepth())), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),image.GetDepth()))]

    inv_transform = transform.GetInverse()

    extreme_points_transformed = [inv_transform.TransformPoint(pnt) for pnt in extreme_points]
    min_x = min(extreme_points_transformed)[0]
    min_y = min(extreme_points_transformed, key=lambda p: p[1])[1]
    min_z = min(extreme_points_transformed, key=lambda p: p[2])[2]
    max_x = max(extreme_points_transformed)[0]
    max_y = max(extreme_points_transformed, key=lambda p: p[1])[1]
    max_z = max(extreme_points_transformed, key=lambda p: p[2])[2]

    # Use the original spacing (arbitrary decision).
    output_spacing = image.GetSpacing()
    # Identity cosine matrix.   
    output_direction = [1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0]
    # Minimal x,y coordinates are the new origin.
    output_origin = [min_x, min_y, min_z]
    
    # Compute grid size based on the physical size and spacing.
    output_size = image.GetSize()
    
    rotated_image = sitk.Resample(image, output_size, transform, sitk.sitkLinear, output_origin, output_spacing,
                                  output_direction)
    return rotated_image

In [ ]:
def circular_mask(cen, rad, shape):
    """
    Generates a 2D binary mask containing a circle.
    
    Parameters
    ----------
    cen : tuple or list of int
        The (row, column) coordinates defining the center of the circle.
    rad : float or int
        The radius of the circle.
    shape : tuple of int
        The (height, width) dimensions of the desired output mask array.

    Returns
    -------
    mask : numpy.ndarray
        A 2D binary array of the specified shape. Pixels falling strictly inside 
        the circle are set to 1, and all others are 0.
    """
    # Create column and row indices representing the grid
    Y, X = np.ogrid[:shape[0], :shape[1]]
    
    # Calculate the squared distance from the center for all pixels simultaneously
    dist_from_center_sq = (Y - int(cen[0]))**2 + (X - int(cen[1]))**2
    
    # Create the mask where the condition is met and convert boolean to integer
    mask = (dist_from_center_sq < rad**2).astype(np.uint8)
    
    return mask

In [ ]:
def ellipse_mask(cen, a, b, shape):
    """
    Generates a 2D binary mask containing an ellipse.
    
    Parameters
    ----------
    cen : tuple or list of int
        The (row, column) coordinates defining the center of the ellipse.
    a : float or int
        The semi-axis length along the row dimension (vertical).
    b : float or int
        The semi-axis length along the column dimension (horizontal).
    shape : tuple of int
        The (height, width) dimensions of the desired output mask array.

    Returns
    -------
    mask : numpy.ndarray
        A 2D binary array of the specified shape. Pixels falling inside or 
        on the boundary of the ellipse are set to 1, and all others are 0.
    """
    mask = np.zeros(shape)
    for row in range(shape[0]):
        for col in range(shape[1]):
            if (row - int(cen[0]))**2/a**2 + (col - int(cen[1]))**2/b**2 <= 1:
                mask[row, col] = 1
    return mask

In [ ]:
def get_global_local_fullbrain_masks(coronal_plane, global_local_fullbrain_processing_params, visualize = False):
    """
    Generates global and local binary masks for a coronal brain slice to isolate the full brain and ventricles.

    This function applies Gaussian blurring, Otsu's global thresholding, and adaptive 
    local thresholding to segment brain tissue from the background. It divides the image 
    into upper, mid, and lower regions based on pixel percentiles to apply specific 
    morphological operations (closing holes up top, removing vertical connections below). 
    It combines global thresholding in the mid-region with local adaptive thresholding 
    elsewhere to ensure optimal segmentation of large structures like ventricles.

    Parameters
    ----------
    coronal_plane : numpy.ndarray
        The 2D image array of the coronal slice to be processed.
    global_local_fullbrain_processing_params : dict
        Dictionary containing processing thresholds, kernel sizes, and percentile thresholds for adaptive ROI placement
    visualize : bool, optional
        If True, displays intermediate processing steps using matplotlib. Default is False.

    Returns
    -------
    coronal_gauss_blurred : numpy.ndarray
        The initial Gaussian-blurred coronal image.
    coronal_otsu_bin : numpy.ndarray
        The boolean mask generated by Otsu's global thresholding.
    global_local_mask : numpy.ndarray
        The composite mask using Otsu thresholding in the middle and adaptive thresholding elsewhere.
    full_brain_mask_bin : numpy.ndarray
        The binary integer mask (0 or 1) representing the isolated full brain.
    cX : int
        The x-coordinate of the full brain mask's centroid.
    cY : int
        The y-coordinate of the full brain mask's centroid.
    """

    coronal_inital_gaussian_blur = global_local_fullbrain_processing_params["coronal_inital_gaussian_blur"]
    coronal_num_iters_close_fill_holes = global_local_fullbrain_processing_params["coronal_num_iters_close_fill_holes"]
    otsu_close_around_ven_mask_col_start_prcntl = global_local_fullbrain_processing_params["otsu_close_around_ven_mask_col_start_prcntl"]
    otsu_close_around_ven_mask_col_stop_prcntl = global_local_fullbrain_processing_params["otsu_close_around_ven_mask_col_stop_prcntl"]
    otsu_close_around_ven_mask_row_start_prcntl = global_local_fullbrain_processing_params["otsu_close_around_ven_mask_row_start_prcntl"]
    otsu_close_around_ven_mask_row_stop_prcntl = global_local_fullbrain_processing_params["otsu_close_around_ven_mask_row_stop_prcntl"]
    lv_open_num_iters = global_local_fullbrain_processing_params["lv_open_num_iters"]
    coronal_adapt_thresh_size = global_local_fullbrain_processing_params["coronal_adapt_thresh_size"]
    coronal_adapt_thresh_mean_thresh = global_local_fullbrain_processing_params["coronal_adapt_thresh_mean_thresh"]
    
    #remove noise by filtering with a Gaussian filter
    coronal_gauss_blurred = cv.GaussianBlur(np.uint8(coronal_plane),(coronal_inital_gaussian_blur,coronal_inital_gaussian_blur), cv.BORDER_DEFAULT)       

    # =================================================================================================
    # Global thresholding with OTSU
    # =================================================================================================
    otsu_thresh = filters.threshold_otsu(coronal_gauss_blurred)
    coronal_otsu_bin = coronal_gauss_blurred > otsu_thresh   
   
    #Define mid and lower ROIs 
    [cr, cc] = coronal_otsu_bin.nonzero()

    if len(cr) == 0:
        print("Warning: Otsu threshold produced an empty mask.")
        h, w = coronal_plane.shape
        return coronal_gauss_blurred, coronal_otsu_bin, np.zeros((h,w)), np.zeros((h,w)), w//2, h//2
    
    lower_mask = np.zeros(coronal_otsu_bin.shape)
    lower_mask[int(np.percentile(cr,80)):max(cr),:] = 1
    mid_mask = np.zeros(coronal_otsu_bin.shape)
    mid_mask[int(np.percentile(cr,20)):int(np.percentile(cr,80)),:] = 1
    upper_mask = np.zeros(coronal_otsu_bin.shape)
    upper_mask[:int(np.percentile(cr,20)),:] = 1
    
    #for any holes up top 
    upper_part_otsu_closed = np.logical_or(cv.morphologyEx(np.uint8(coronal_otsu_bin*upper_mask), 
                                                                cv.MORPH_CLOSE,
                                                                cv.getStructuringElement(cv.MORPH_ELLIPSE,(coronal_inital_gaussian_blur,
                                                                                                           coronal_inital_gaussian_blur)), 
                                                                iterations = coronal_num_iters_close_fill_holes), coronal_otsu_bin*(1-upper_mask))

    # =================================================================================================
    # Centroid and Ellipse Masking
    # =================================================================================================
    mask_for_full_brain = upper_part_otsu_closed
    mask_for_full_brain = upper_part_otsu_closed
    M0 = cv.moments(np.uint8(mask_for_full_brain))

    if M0["m00"] != 0:
        cX = int(M0["m10"] / M0["m00"])
        cY = int(M0["m01"] / M0["m00"])
    else:
        cX, cY = coronal_plane.shape[1] // 2, coronal_plane.shape[0] // 2
    
    [c_vrt_rm_r,c_vrt_rm_c] = mask_for_full_brain.nonzero()

    if len(c_vrt_rm_r) > 0:
        # We want this ellipse to be smaller in row range so it fits well within the brain   
        coronal_otsu_bin_col_range = int(np.percentile(c_vrt_rm_c, otsu_close_around_ven_mask_col_stop_prcntl) - 
                                         np.percentile(c_vrt_rm_c, otsu_close_around_ven_mask_col_start_prcntl))
        coronal_otsu_bin_row_range = int(np.percentile(c_vrt_rm_r, otsu_close_around_ven_mask_row_stop_prcntl) - 
                                         np.percentile(c_vrt_rm_r, otsu_close_around_ven_mask_row_start_prcntl))          
    else:
        coronal_otsu_bin_col_range, coronal_otsu_bin_row_range = 0, 0

              
    b = coronal_otsu_bin_col_range//2
    a = coronal_otsu_bin_row_range//2
    if coronal_otsu_bin_row_range > coronal_otsu_bin_col_range:
        a, b = b, a

    #using a 10 mm disk
    inner_a = max(0, a - 10)
    inner_b = max(0, b - 10)
    
    around_ventricle_mask = (ellipse_mask((cY, cX), a, b, mask_for_full_brain.shape) - 
                             ellipse_mask((cY, cX), inner_a, inner_b, mask_for_full_brain.shape))
    
    closed_coronal_for_full_brain_mask = np.logical_or(around_ventricle_mask,mask_for_full_brain)
    
    #This is to get a full brain mask
    im_thf = closed_coronal_for_full_brain_mask.copy()
    hf, wf = im_thf.shape[:2]
    maskf = np.zeros((hf+2, wf+2), np.uint8)
    im_floodfillf = im_thf.copy()
    # Floodfill from point (0, 0)
    resf = cv.floodFill(np.uint8(im_floodfillf), maskf, (0,0), 255)
    full_brain_mask_f = (1-resf[2]).copy()[1:hf+1,1:wf+1]

    lower_con_rem_kernel = np.array([[1,1,1],[0,1,0],[1,1,1]],'uint8')
    
    lower_part_con_removed = np.logical_or(full_brain_mask_f*(1-lower_mask),
                                     cv.morphologyEx(np.uint8(full_brain_mask_f*
                                                                lower_mask), 
                                                    cv.MORPH_OPEN,
                                                    lower_con_rem_kernel,
                                                    iterations = lv_open_num_iters))
    
    
    full_brain_mask = getLargestCC(measure.label(lower_part_con_removed, background = 0))
    
    # ==================================================================================================
    # Adaptive Thresholding & Final Mask Generation
    # ==================================================================================================
    [fbnr,fbnc] = (1-full_brain_mask).nonzero()
    coronal_adapt_thresh = cv.adaptiveThreshold(coronal_gauss_blurred,1,cv.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv.THRESH_BINARY,coronal_adapt_thresh_size,coronal_adapt_thresh_mean_thresh)
    coronal_adapt_bg_suppressed = coronal_adapt_thresh.copy()
    coronal_adapt_bg_suppressed[fbnr,fbnc] = 0
    

    #Use the global thresholded image in the middle ROI because we want to capture all of the ventricles
    #which may not be captured by the adaptive threshold to follow when they are too big. Retain adaptive thresh 
    #up top (for closed holes) and below. Because when we capture the difference between the adaptive and otsu 
    #masks, only the middle portion which matters is highlighted. 

    global_local_mask = np.logical_or(coronal_otsu_bin*mid_mask,
                                               coronal_adapt_bg_suppressed*(1-mid_mask))
    
    
    if visualize:
        plt.imshow(coronal_plane, cmap = 'gray')
        plt.title('Coronal plane')
        plt.show()
        plt.imshow(coronal_gauss_blurred,cmap = 'gray')
        plt.title('Coronal gauss blurred')
        plt.show()
        plt.imshow(coronal_otsu_bin,cmap = 'gray')
        plt.title('Coronal OTSU')
        plt.show()
        plt.imshow(mask_for_full_brain,cmap = 'gray')
        plt.title('Coronal bin, closed up top and vertical connections removed at bottom')
        plt.show()
        plt.imshow(full_brain_mask,cmap = 'gray')
        plt.title('Full brain mask')
        plt.show()
        plt.imshow(global_local_mask,cmap = 'gray')
        plt.title('Local thresholding up top and below + OTSU in the middle to capture whole ventricles when they are too large')
        plt.show()

    return coronal_gauss_blurred, coronal_otsu_bin, global_local_mask, np.where(full_brain_mask,1,0), cX, cY

In [ ]:
def get_ventricles_adaptive(coronal_gauss_blurred, full_brain_mask, adaptive_ventricles_processing_parameters, visualize = False):

    """
    Isolates the ventricles using an iteratively relaxing adaptive threshold.

    This function applies adaptive Gaussian thresholding to a blurred coronal slice. 
    It compares the resulting segmented ventricle area to the total brain area. If the 
    ratio of ventricle-to-brain is below a defined threshold, it iteratively reduces 
    the adaptive threshold constant to capture more ventricular area. 

    Parameters
    ----------
    coronal_gauss_blurred : numpy.ndarray
        The 2D image array of the Gaussian-blurred coronal slice.
    full_brain_mask : numpy.ndarray
        The binary mask (0s and 1s) representing the isolated full brain.
    adaptive_ventricles_processing_parameters : dict
        Dictionary containing processing thresholds and constants.
    visualize : bool, optional
        If True, displays the intermediate background-suppressed and final ventricle 
        masks using matplotlib. Default is False.

    Returns
    -------
    ventricle_mask_from_adaptive_0 : numpy.ndarray
        The binary mask of the segmented ventricles. Returns an empty mask (all zeros) 
        if the input full_brain_mask is empty.
    """

    total_brain_area = np.sum(full_brain_mask)
    if total_brain_area == 0:
        print("full_brain_mask is empty. Adaptive thresholding failed.")
        return np.zeros_like(full_brain_mask)
    
    ventricle_brain_ratio_thresh = adaptive_ventricles_processing_parameters["ventricle_brain_ratio_thresh"]
    constant_for_adaptive_thresh = adaptive_ventricles_processing_parameters["constant_for_adaptive_thresh"]
    adaptive_i = adaptive_ventricles_processing_parameters["adaptive_i"]
    coronal_initial_adaptive_thresh_size = adaptive_ventricles_processing_parameters["coronal_initial_adaptive_thresh_size"]
    coronal_adaptive_thresh_size = adaptive_ventricles_processing_parameters["coronal_adaptive_thresh_size"]


    coronal_adapt_thresh = cv.adaptiveThreshold(np.uint8(coronal_gauss_blurred),1,cv.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv.THRESH_BINARY, coronal_initial_adaptive_thresh_size, constant_for_adaptive_thresh - adaptive_i) 
    coronal_adapt_thresh_bg_supressed = np.where((1-full_brain_mask)>0,0,coronal_adapt_thresh)
    ventricle_mask_from_adaptive_0 = full_brain_mask - coronal_adapt_thresh_bg_supressed
    ventricle_brain_ratio = np.sum(ventricle_mask_from_adaptive_0)/np.sum(full_brain_mask)
    
    while True:
        if (ventricle_brain_ratio >= ventricle_brain_ratio_thresh):        
            break
        else: #reducing the constant for adaptive thresh to capture more ventricular area 
            if (adaptive_i > 7):
                break
            adaptive_i = adaptive_i + 1
            coronal_adapt_thresh = cv.adaptiveThreshold(coronal_gauss_blurred,1,cv.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv.THRESH_BINARY,coronal_adaptive_thresh_size,constant_for_adaptive_thresh - adaptive_i)
            coronal_adapt_thresh_bg_supressed = np.where((1-full_brain_mask)>0,0,coronal_adapt_thresh)
            ventricle_mask_from_adaptive_0 = (full_brain_mask - coronal_adapt_thresh_bg_supressed)
            ventricle_brain_ratio = np.sum(ventricle_mask_from_adaptive_0)/total_brain_area
            
    if visualize:
        plt.imshow(coronal_adapt_thresh_bg_supressed,cmap = 'gray')
        plt.title('Coronal background suppressed adaptive')
        plt.show()
        plt.imshow(ventricle_mask_from_adaptive_0,cmap = 'gray')
        plt.title('Ventricle mask from adaptive, adjusted to capture ventricles even when they are small')
        plt.show()
    return ventricle_mask_from_adaptive_0

In [ ]:
def get_middle_boat_shaped_roi(ventricle_mask_from_adaptive_0, cX, cY, visualize = False):

    """
    Generates a  boat-shaped (V-on-rectangle) Region of Interest (ROI).

    This ROI is designed to dynamically follow the ventricle boundaries while narrowing
    near the center to prevent accidental leakage/connection into the brain midline.
    The function calculates the adaptive ventricle centroid and forms a symmetric 
    geometric mask around the provided brain center coordinates.

    Parameters
    ----------
    ventricle_mask_from_adaptive_0 : numpy.ndarray
        2D binary mask representing the ventricles isolated from adaptive thresholding.
    cX : int
        The x-coordinate (column) of the full brain mask's centroid.
    cY : int
        The y-coordinate (row) of the full brain mask's centroid.
    visualize : bool, optional
        If True, overlays the generated boat-shaped ROI onto the ventricle mask 
        along with the calculated centroid using matplotlib. Default is False.

    Returns
    -------
    middle_roi_0 : numpy.ndarray
        2D binary mask containing the boat-shaped region of interest.
    vcX : int
        The x-coordinate of the adaptive ventricle centroid.
    vcY : int
        The y-coordinate of the adaptive ventricle centroid.
    """
    if np.sum(ventricle_mask_from_adaptive_0) == 0:
        print("Warning: Adaptive ventricle mask is empty. Returning blank ROI.")
        return np.zeros_like(ventricle_mask_from_adaptive_0), cX, cY
        
    #boat-shaped so that we don't accidentally connect with midline 
    Mv = cv.moments(np.uint8(ventricle_mask_from_adaptive_0))
    [vra,vca] = ventricle_mask_from_adaptive_0.nonzero()
    
    #Column range for middle roi
    half_col_range_mr = int((np.percentile(vca,80) - np.percentile(vca,20))//2)
    vcX = int(np.percentile(vca,50))
    
    if Mv["m00"] != 0:
        vcY = int(Mv["m01"] / Mv["m00"])
    else:
        vcY = cY
        
    h, w = ventricle_mask_from_adaptive_0.shape
    middle_roi_0 = np.zeros((h, w))
    
    #Here we define a v-on-top-of-a-rect shaped ROI
    r_start = max(0, cY - 5)
    r_end = min(h, cY + 15)
    c_start = max(0, cX - half_col_range_mr)
    c_end = min(w, cX)
    
    # Define a v-on-top-of-a-rect shaped ROI (Left side first)
    middle_roi_0[r_start:r_end, c_start:c_end] = 1
    
    upper_half_row_range = range(max(0, cY - 5), min(h, cY + 5))
    c_range = range(max(0, cX - half_col_range_mr), min(w, cX))
    
    # Maintain theoretical slope threshold using original intended offsets
    m_thresh = (cY - 5) - (cX - half_col_range_mr)
    
    for rm in upper_half_row_range:
        for cm in c_range:
            if (cm > (rm-m_thresh)):
                middle_roi_0[rm,cm] = 0

    left_processed_width = c_end - c_start
    dest_c_start = cX
    dest_c_end = min(w, cX + left_processed_width)
    
    actual_mirror_width = dest_c_end - dest_c_start
    
    if actual_mirror_width > 0:
        # Match the left source window width exactly to the available right destination width
        middle_roi_0[:, dest_c_start:dest_c_end] = middle_roi_0[:, cX - actual_mirror_width:cX][:, ::-1]
    
    if visualize:
        plt.imshow(ventricle_mask_from_adaptive_0, cmap = 'gray')
        plt.imshow(middle_roi_0, alpha = 0.2)
        plt.scatter(vcX, vcY, marker = 'x')
        plt.title('Middle boat shaped ROI for processing, with adaptive ventricle centroid')
        plt.show()
        
    return middle_roi_0, vcX, vcY

In [ ]:
def get_ventricles_global_local_fused(ventricle_mask_from_otsu_0, ventricle_mask_from_adaptive_0, vcY, visualize = False):

    """
    Fuses global (Otsu) and local (Adaptive) ventricle masks based on spatial and area heuristics.

    This function attempts to fill in missing parts of the adaptive mask by looking at the 
    Otsu mask. It isolates the regions captured by Otsu but missed by adaptive thresholding. 
    If the largest continuous chunk of this difference is close enough to the expected 
    ventricle centroid (Y-axis) and makes up a significant portion of the total Otsu area, 
    it is merged into the final adaptive mask.

    Parameters
    ----------
    ventricle_mask_from_otsu_0 : numpy.ndarray
        2D binary mask of the ventricles generated via Otsu's global thresholding.
    ventricle_mask_from_adaptive_0 : numpy.ndarray
        2D binary mask of the ventricles generated via adaptive local thresholding.
    vcY : int
        The Y-coordinate (row) of the adaptive ventricle centroid.
    visualize : bool, optional
        If True, displays the fusion contributions and final merged mask using 
        matplotlib. Default is False.

    Returns
    -------
    ventricle_mask_from_adaptive_1 : numpy.ndarray
        The finalized 2D binary mask containing the fused ventricle segmentation.
    """
    
    # Initialize the output mask as the baseline adaptive mask
    ventricle_mask_from_adaptive_1 = ventricle_mask_from_adaptive_0.copy()
    # We only want pixels that are TRUE in Otsu but FALSE in Adaptive
    diff_mask_raw = np.logical_and(ventricle_mask_from_otsu_0 > 0, ventricle_mask_from_adaptive_0 == 0)

    # Check if there is actually any difference to process
    if np.sum(diff_mask_raw) > 0:
        
        # Get the largest connected component of the difference
        diff_otsu_adap = getLargestCC(measure.label(diff_mask_raw, background=0))  
        
        Md = cv.moments(np.uint8(diff_otsu_adap))
        total_otsu_area = np.sum(ventricle_mask_from_otsu_0)
        

        if Md["m00"] != 0 and total_otsu_area > 0:
            dX = int(Md["m10"] / Md["m00"])
            dY = int(Md["m01"] / Md["m00"])
            
            diff_area_ratio = np.sum(diff_otsu_adap) / total_otsu_area

            # If the chunk is close to the vertical center and reasonably large, fuse it
            if (vcY - dY < 7) and (diff_area_ratio > 0.1):
                ventricle_mask_from_adaptive_1 = np.logical_or(ventricle_mask_from_adaptive_0, diff_otsu_adap)
                
                if visualize:
                    fig, ax = plt.subplots(figsize=(6, 6))
                    ax.imshow(ventricle_mask_from_adaptive_0, cmap='gray')
                    ax.imshow(diff_otsu_adap, cmap='hot', alpha=0.4)
                    ax.set_title('Contribution of Otsu to Adaptive (Highlighted)')
                    plt.show()

    if visualize:
        plt.figure(figsize=(6, 6))
        plt.imshow(ventricle_mask_from_adaptive_1, cmap='gray')
        plt.title('Adaptive and Otsu Fused Ventricles')
        plt.show()
        
    return ventricle_mask_from_adaptive_1

In [ ]:
def smooth_processed_ventricles(ventricles, full_brain_mask, visualize = False):
    """
    Applies an adaptive median blur to smooth the boundaries of the isolated ventricles.

    To avoid entirely erasing small ventricles, the function checks the ratio of the 
    smoothed ventricles to the total brain area. If the ratio drops below 1%, it 
    iteratively reduces the blur kernel size (from 5, to 3, to 1) until the ratio is 
    acceptable or the minimum kernel size is reached.

    Parameters
    ----------
    ventricles : numpy.ndarray
        2D binary mask of the segmented ventricles.
    full_brain_mask : numpy.ndarray
        2D binary mask representing the isolated full brain.
    visualize : bool, optional
        If True, displays the final smoothed ventricle mask using matplotlib. 
        Default is False.

    Returns
    -------
    ventricles_smoothed : numpy.ndarray
        The 2D binary mask of the smoothed ventricles.
    ventricle_smoothed_brain_ratio : float
        The ratio of the smoothed ventricle area to the total brain area.
    """
    total_brain_area = np.sum(full_brain_mask)
    if total_brain_area == 0:
        print("Warning: full_brain_mask is empty. Skipping smoothing.")
        return np.zeros_like(ventricles), 0.0

    ventricles_8u = np.uint8(ventricles)
    
    smoothing_factor, sf_i = 5, 0

    k_size = max(1, smoothing_factor - sf_i)
    if k_size % 2 == 0: k_size += 1
    
    ventricles_smoothed = cv.medianBlur(np.uint8(ventricles_8u),k_size)
    ventricle_smoothed_brain_ratio = np.sum(ventricles_smoothed)/total_brain_area
    
    while True:
        if  ventricle_smoothed_brain_ratio >= 0.01:
            break
        else:
            if (sf_i > 3):
                # Lowering the threshold did not work; exit the loop
                break
            sf_i = sf_i + 2
            
            k_size = max(1, smoothing_factor - sf_i)
            if k_size % 2 == 0: k_size += 1
            
            ventricles_smoothed = cv.medianBlur(ventricles_8u, k_size)
            ventricle_smoothed_brain_ratio = np.sum(ventricles_smoothed) / total_brain_area
            
    if visualize:
        plt.imshow(ventricles_smoothed,cmap = 'gray')
        plt.title('Smoothed ventricles')
        plt.show()
    return ventricles_smoothed, ventricle_smoothed_brain_ratio

In [ ]:
def num_connected_comps_cv(clean_ventricles, area_check_thresh = 0.95, disconnectedLV_x_diff_tol = 20, visualize = False): 
    """
    Filters and cleans connected components in the anterior ventricle sections.

    This function isolates the most likely ventricular blobs by evaluating their 
    size, vertical row variation, and relative distance to one another. It is 
    specifically designed to prevent artifacts, such as the longitudinal fissure, 
    from being mistakenly included in the final mask.

    Parameters
    ----------
    clean_ventricles : numpy.ndarray
        2D binary mask of the isolated ventricles prior to component filtering.
    visualize : bool, optional
        If True, displays the final processed blobs using matplotlib. Default is False.

    Returns
    -------
    processed_blobs : numpy.ndarray
        The finalized 2D binary mask containing only the validated ventricular components.
    """
    blobs_labels = measure.label(clean_ventricles, background=0)
    unique_labels = np.unique(blobs_labels)

    if len(unique_labels) <= 1:
        print("Warning: No connected components found. Returning empty mask.")
        return False, np.zeros_like(clean_ventricles)

    processed_blobs = blobs_labels.copy()

    # --- PHASE 1: Downselect to the top 3 blobs (if more than 2 blobs exist) ---
    if len(unique_labels) > 3: 
        bincounts = np.bincount(blobs_labels.flatten())
        bincounts[0] = 0  # Ignore background for size sorting
        top_3_labels = np.argsort(bincounts)[::-1][:3]

        mask1 = (blobs_labels == top_3_labels[0])
        mask2 = (blobs_labels == top_3_labels[1])
        mask3 = (blobs_labels == top_3_labels[2])

        pr1, pc1 = np.where(mask1)
        pr2, pc2 = np.where(mask2)
        pr3, pc3 = np.where(mask3)


        if len(pr1) > 0 and len(pr2) > 0 and len(pr3) > 0:
            closest_inds = np.argmin([
                np.abs(np.median(pr1) - np.median(pr2)), 
                np.abs(np.median(pr1) - np.median(pr3)),
                np.abs(np.median(pr2) - np.median(pr3))
            ])
            
            var_ranges = [np.max(pr1) - np.min(pr1), np.max(pr2) - np.min(pr2), np.max(pr3) - np.min(pr3)]
            blob_sizes = [np.sum(mask1), np.sum(mask2), np.sum(mask3)]
            
            # Cascade logic to filter out the most unlikely blob
            if blob_sizes[0] < 0.1 * max(blob_sizes[1], blob_sizes[2]):
                processed_blobs = np.where(mask2 | mask3, blobs_labels, 0)
            elif blob_sizes[1] < 0.1 * max(blob_sizes[0], blob_sizes[2]):
                processed_blobs = np.where(mask1 | mask3, blobs_labels, 0)
            elif blob_sizes[2] < 0.1 * max(blob_sizes[1], blob_sizes[0]):
                processed_blobs = np.where(mask1 | mask2, blobs_labels, 0)
            else:
                if var_ranges[0] > 1.5 * max(var_ranges[1], var_ranges[2]):
                    processed_blobs = np.where(mask2 | mask3, blobs_labels, 0)
                elif var_ranges[1] > 1.5 * max(var_ranges[0], var_ranges[2]):
                    processed_blobs = np.where(mask1 | mask3, blobs_labels, 0)
                elif var_ranges[2] > 1.5 * max(var_ranges[0], var_ranges[1]):
                    processed_blobs = np.where(mask1 | mask2, blobs_labels, 0)
                else:            
                    if closest_inds == 0:
                        processed_blobs = np.where(mask1 | mask2, blobs_labels, 0)
                    elif closest_inds == 1:
                        processed_blobs = np.where(mask1 | mask3, blobs_labels, 0)
                    else:
                        processed_blobs = np.where(mask2 | mask3, blobs_labels, 0)
        else:
            # Fallback if a blob is somehow empty: keep the largest 2 valid masks
            processed_blobs = np.where(mask1 | mask2, blobs_labels, 0)

    # --- PHASE 2: Remap labels cleanly to 0, 1, 2 ---
    rem_labels = np.unique(processed_blobs)
    mapped_blobs = np.zeros_like(processed_blobs)
    for new_val, old_val in enumerate(rem_labels):
        mapped_blobs[processed_blobs == old_val] = new_val
    processed_blobs = mapped_blobs


    # --- PHASE 3: Evaluate Background + 2 remaining blobs ---
    if len(np.unique(processed_blobs)) == 3: 
        counts = np.bincount(processed_blobs.flatten())
        sorted_args = np.argsort(counts)
        smallest_blob_arg = sorted_args[0]
        mid_blob_arg = sorted_args[1]

        sb_r, sb_c = np.where(processed_blobs == smallest_blob_arg)
        mb_r, mb_c = np.where(processed_blobs == mid_blob_arg)
        pr, pc = np.where(processed_blobs > 0)

        if len(sb_r) > 0 and len(mb_r) > 0 and len(pc) > 0:
            im_cen = (np.max(pc) + np.min(pc)) // 2
            
            smallest_blob_height = (np.min(sb_r) + np.max(sb_r)) // 2
            smallest_blob_cen_x = (np.min(sb_c) + np.max(sb_c)) // 2

            mid_blob_height = (np.min(mb_r) + np.max(mb_r)) // 2
            mid_blob_cen_x = (np.min(mb_c) + np.max(mb_c)) // 2

            nv_mass = (smallest_blob_height < np.min(mb_r)) | (smallest_blob_height > np.max(mb_r))
            area_check = (len(mb_r) - len(sb_r)) > area_check_thresh * len(mb_r)

            if ((np.abs(smallest_blob_cen_x - im_cen) <= disconnectedLV_x_diff_tol) & (np.abs(mid_blob_cen_x - im_cen) <= disconnectedLV_x_diff_tol) &
                (~nv_mass) and (~area_check)):
                pass
            else: 
                # Remove the smallest blob if it fails the criteria
                processed_blobs[processed_blobs == smallest_blob_arg] = 0

    # Final binarization
    processed_blobs = np.where(processed_blobs > 0, 1, 0) 
     
    if visualize:
        plt.figure(figsize=(6, 6))
        plt.imshow(processed_blobs, cmap='gray')
        plt.title("Processed Ventricle Blobs (Artifacts Removed)")
        plt.show()
      
    return processed_blobs

##### Data setup

###### Before running the pipeline, make a copy of config.template.json, rename it to config.json, and update the paths to point to your local dataset.

In [ ]:
# Load the configuration file
with open('config.json', 'r') as file:
    config = json.load(file)

# Assign variables dynamically
data_path = config['data_path']
coronal_acpc_vol_aligned_name = config['coronal_acpc_vol_aligned_name']
info_csv_path = config['info_csv_path_w_name']
scan_id_col_name = config['scan_id_col_name']
pat_id_col_name = config['pat_id_col_name']
ac_coordinates_col_name = config['ac_coordinates_col_name']
pc_coordinates_col_name = config['pc_coordinates_col_name']

output_folder = config['output_csv_write_folder']

print(f"Loading data from: {data_path}")

In [ ]:
info_df = pd.read_csv(info_csv_path)
info_df[scan_id_col_name] = info_df[scan_id_col_name].str.strip("'")

In [ ]:
accnum_list = info_df[scan_id_col_name].values

In [ ]:
len(accnum_list), len(info_df)

In [ ]:
#dataframe which will contain track of the computed EI-z values for each scan
ei_z_df = pd.DataFrame()

###### Predefined, emperically determined processing parameters corresponding to ROI sizes, positions, and global/local thresholding parameters. 
The following parameters work for a diverse cohort of chronic neurodegenerative conditions spanning Normal Pressure Hydrocephalus, Alzheimer's disease, post-traumatic encephalomalacia, and headache. So they are recommended to be used directly for patient scans without acute pathology like significant midline shifts, bleeds, or mass effects. 

In [ ]:
global_local_fullbrain_processing_params = dict()
global_local_fullbrain_processing_params["coronal_inital_gaussian_blur"] = 3
global_local_fullbrain_processing_params["coronal_num_iters_close_fill_holes"] = 7
global_local_fullbrain_processing_params["otsu_close_around_ven_mask_col_start_prcntl"] = 5
global_local_fullbrain_processing_params["otsu_close_around_ven_mask_col_stop_prcntl"] = 95
global_local_fullbrain_processing_params["otsu_close_around_ven_mask_row_start_prcntl"] = 15
global_local_fullbrain_processing_params["otsu_close_around_ven_mask_row_stop_prcntl"] = 85
global_local_fullbrain_processing_params["lv_open_num_iters"] = 3
global_local_fullbrain_processing_params["coronal_adapt_thresh_size"] = 85
global_local_fullbrain_processing_params["coronal_adapt_thresh_mean_thresh"] = 7.5

In [ ]:
adaptive_ventricles_processing_parameters = dict()
adaptive_ventricles_processing_parameters["ventricle_brain_ratio_thresh"] = 0.01
adaptive_ventricles_processing_parameters["constant_for_adaptive_thresh"] = 7.5
adaptive_ventricles_processing_parameters["adaptive_i"] =  0    
adaptive_ventricles_processing_parameters["coronal_initial_adaptive_thresh_size"] = 95
adaptive_ventricles_processing_parameters["coronal_adaptive_thresh_size"] = 85

##### EI-z pipeline

In [ ]:
ei_z_df = pd.DataFrame()

In [ ]:
for acc_num in accnum_list:
    
    pat = info_df[pat_id_col_name][info_df[scan_id_col_name]==acc_num].values[0] 

    #Get AC coordinates
    aca_str = info_df[ac_coordinates_col_name][info_df[scan_id_col_name]==acc_num].values[0]
    [x,y,z] = aca_str.split(",")
    aca_x,aca_y,aca_z = [float(x.strip("[")),float(y),float(z.strip("]"))]      

    print(f'Patient is {pat} and acc_num is {acc_num}')   
       
    cor_img_path = os.path.join(data_path, acc_num, coronal_acpc_vol_aligned_name)

    ########## Read in coronal volume 
    cor_img = sitk.ReadImage(cor_img_path)
    cor_img_brain = window_stack_sitk(rotate_image(cor_img),40,80)

    ac = [aca_x,aca_y,aca_z]

    cor_img_brain_dat = sitk.GetArrayFromImage(cor_img_brain)
    cor_ac_index = cor_img_brain.TransformPhysicalPointToContinuousIndex(ac)
    pl  = int(np.round(cor_ac_index[1]))
    
    max_fhh = 0

    #Evaluate from AC location to 20 mm inferiorly to AC
    for plane in range(pl-20,pl): 
        coronal_plane = cor_img_brain_dat[:,plane,:]          
        
        coronal_gauss_blurred, coronal_bin, global_local_mask, full_brain_mask, cX, cY = get_global_local_fullbrain_masks(coronal_plane, 
                                                                                                      global_local_fullbrain_processing_params)
        
        #Ventricles from Adaptive
        ventricle_mask_from_adaptive_0 = get_ventricles_adaptive(coronal_gauss_blurred, full_brain_mask, adaptive_ventricles_processing_parameters) 
        middle_roi_0, vcX, vcY = get_middle_boat_shaped_roi(ventricle_mask_from_adaptive_0, cX, cY)            
        clean_ven_adaptive = np.logical_or(middle_roi_0, ventricle_mask_from_adaptive_0)        
        blobs_labels_adaptive = measure.label(clean_ven_adaptive, background = 0)
        largest_cc_ven_adaptive = getLargestCC(blobs_labels_adaptive)      
        [row_a, col_a] = (1-largest_cc_ven_adaptive).nonzero()
        ventricle_mask_from_adaptive_0[row_a, col_a] = 0
        
        #Ventricles from OTSU (Middle), Adaptive (elsewhere)
        ventricle_mask_from_otsu_0 = full_brain_mask - global_local_mask
        clean_ven_otsu = np.logical_or(middle_roi_0, ventricle_mask_from_otsu_0)        
        blobs_labels_otsu = measure.label(clean_ven_otsu, background = 0)
        largest_cc_ven_otsu = getLargestCC(blobs_labels_otsu)      
        [row_o, col_o] = (1-largest_cc_ven_otsu).nonzero()
        ventricle_mask_from_otsu_0[row_o, col_o] = 0
        
        #Check for difference between adaptive and otsu (in terms of y-centroid and mass)
        if np.sum(ventricle_mask_from_otsu_0) > 0:            
            ventricle_mask_from_adaptive_1 = get_ventricles_global_local_fused(ventricle_mask_from_otsu_0, 
                                                                               ventricle_mask_from_adaptive_0, 
                                                                               vcY)
        else: ventricle_mask_from_adaptive_1=ventricle_mask_from_adaptive_0
        
        pt = int(coronal_plane.shape[0]-cor_ac_index[2]) 
        above_ac_mask = np.zeros(coronal_plane.shape)
        above_ac_mask[0:pt,:] = 1
        ventricles = np.where(ventricle_mask_from_adaptive_1*above_ac_mask>0,1,0)

        #Smooth LVs in the required ROI
        ventricles_smoothed, ventricle_smoothed_brain_ratio = smooth_processed_ventricles(ventricles, full_brain_mask)
        
        if ventricle_smoothed_brain_ratio < 0.01:
            continue
            
        [nrs,ncs] = ventricles_smoothed.nonzero()      
        
        if (max(nrs) - min(nrs)) >  1.5*(max(ncs) - min(ncs)):
            continue

        #Further LV clean up
        try:
            ventricles_final = num_connected_comps_cv(ventricles_smoothed)
        except AssertionError:
            continue
            
        #Identify required contour points for LV height measurement on processed plane
        [nr,nc] =  ventricles_final.nonzero()
        mvc = (max(nc)+min(nc))//2
        
                
        if np.sum(ventricles_final)/np.sum(full_brain_mask) < 0.01:
            continue

        sorted_cols = sorted(nc)
        sorted_cols_left = [x for x in sorted_cols if x <= mvc-2]
        sorted_cols_right = [x for x in sorted_cols if x > mvc+2]
        top_rows = [min(nr[nc==c]) for c in sorted_cols]
        
        #small offset incase there is ML atrophy
        left_rows = np.array(top_rows)[np.where(sorted_cols <= mvc-2)[0]]
        right_rows = np.array(top_rows)[np.where(sorted_cols > mvc+2)[0]]

        sorted_unique_cols = np.unique(sorted_cols)
        opt_ly1 = min(left_rows)
        opt_ly2 = pt
        max_l_dist = opt_ly2 - opt_ly1
        opt_lx = min(np.array(sorted_cols_left)[np.where(left_rows==opt_ly1)[0]])
        
        opt_ry1 = min(right_rows)
        opt_ry2 = pt
        max_r_dist = opt_ry2 - opt_ry1
        opt_rx = max(np.array(sorted_cols_right)[np.where(right_rows==opt_ry1)[0]])


        fhh = max(max_l_dist, max_r_dist)
        left_flag = 0
        if max_l_dist > max_r_dist:
            left_flag = 1
        
        if fhh > max_fhh:
            max_fhh = fhh
            opt_left_flag = left_flag
            opt_pl = plane
            opt_full_brain_mask = full_brain_mask
            opt_coronal_plane = coronal_plane
            opt_coronal_bin = coronal_bin
            opt_opt_lx = opt_lx
            opt_opt_rx = opt_rx
            opt_opt_ly1 = opt_ly1
            opt_opt_ly2 = opt_ly2
            opt_opt_ry1 = opt_ry1
            opt_opt_ry2 = opt_ry2
            opt_sorted_unique_cols = sorted_unique_cols


    if max_fhh == 0:
        print(f'Could not process patient {pat} and acc_num {acc_num}')
    else: 
        [fnr, fnc] = (opt_full_brain_mask*opt_coronal_bin).nonzero()
        sorted_unique_bcols = sorted(np.unique(fnc))
        selected_fcols = [c for c in sorted_unique_bcols if c <= np.min(opt_sorted_unique_cols)
                          or c >= np.max(opt_sorted_unique_cols)]

        max_bl = 0
        for col in selected_fcols: 
            fy1 = min(fnr[fnc == col])
            fy2 = max(fnr[fnc == col])
            cur_bl = fy2 - fy1
            if cur_bl > max_bl:
                max_bl = cur_bl
                fy1_opt, fy2_opt = fy1, fy2
                fx_opt = col


        fig, ax = plt.subplots(figsize = (12,12))
        ax.imshow(opt_coronal_plane,cmap = 'gray')

        if opt_left_flag == 1:
            ax.scatter([opt_opt_lx]*(opt_opt_ly2-opt_opt_ly1+1), [x for x in range(opt_opt_ly1, opt_opt_ly2+1)],
                       marker = 'x', s = 2, color = 'w')
        else: 
            ax.scatter([opt_opt_rx]*(opt_opt_ry2-opt_opt_ry1+1), [x for x in range(opt_opt_ry1, opt_opt_ry2+1)],
                   marker = 'x', s = 2, color = 'w')
        ax.scatter([fx_opt]*(fy2_opt-fy1_opt+1), [x for x in range(fy1_opt, fy2_opt+1)], marker = 'x', s = 2, 
                   color = 'w')
        plt.show()

        if max_bl > 0:
            ei_z = max_fhh/(max_bl)        
            ei_z_df = pd.concat([ei_z_df, pd.DataFrame({pat_id_col_name:[pat],scan_id_col_name:["'"+acc_num+"'"],'ei_z':[ei_z]
                                                       })], ignore_index = True)
        else:
            print(f'Could not process patient {pat} and acc_num {acc_num}')
   
        


In [ ]:
len(ei_z_df), sum(ei_z_df['ei_z'].isna())

In [ ]:
#Examine histogram of computed values
plt.hist(ei_z_df['ei_z'])
plt.xticks([x for x in np.arange(0, 0.7, 0.1)])
plt.show()

In [ ]:
#Check for any extreme values and inspect the corresponding outputs above for computational errors
min_ei_z_lim = 0.1
max_ei_z_lim = 0.6

ei_z_df[(ei_z_df['ei_z'] <= min_ei_z_lim) | (ei_z_df['ei_z'] >= max_ei_z_lim)]

In [ ]:
ei_z_df.to_csv(os.path.join(output_folder, 'ei_z.csv'))